# Suite CNT — Containers

A container holds elements by index (`Schemas.OfContainer`): a property's value, read and written through
`Visitors.OfContainer`, its elements each through a `Visitors.OfAny`. Dense, its elements are a packed array; sparse,
blocks of them; with neither, it is dense for an integer-like index and a dict for any other hashable one. These cases
cover proxies, iterators, the value objects containers hold, and snapshots.

In [ ]:
from mbse.Schemas.Framework import JSON, Plain, Proxies, Schemas as S, YAML
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import raises, root_of, text

Int, Text, Float = S.OfScalar.Data(int), S.OfScalar.Data(str), S.OfScalar.Data(float)
Phone = S.OfObject.Builder().properties(text("number"), text("label")).create()
Cell = S.OfObject.Builder().properties(text("row", int), text("col", int)).create()
Tags = S.OfContainer.Builder().index(Int).element(Text).create()  # dense
Phones = S.OfContainer.Builder().index(Text).element(Phone).create()  # a dict
Grid = S.OfContainer.Builder().index(Cell).element(Float).create()  # dense, of two indices
Sparse = S.OfContainer.Builder().index(Int).element(Text).sparse().create()
Rows = S.OfContainer.Builder().index(Int).element(Tags).create()  # nested
Card = S.OfObject.Builder().name("Card").ref(r"card-(\d+)").properties(
    lambda p: p.name("tags").of(Tags), lambda p: p.name("phones").of(Phones), lambda p: p.name("grid").of(Grid),
    lambda p: p.name("sparse").of(Sparse), lambda p: p.name("rows").of(Rows), text("name")).create()
store = Proxies.OfStore()
store.register(Card)


def container(read):
    """The container a property holds, as its visitor; None where it holds none (its visitor then visits nothing)."""
    found = []
    read.as_value(lambda a: a.as_container(lambda c: found.append(c) or c))
    return found[0] if found[0].as_schema() is not None else None


def scalars(c):
    """A container's elements present, each with its index, as scalars."""
    out = []
    c.as_entries(lambda pairs: out.extend((index, value_of_any(e)) for index, e in pairs))
    return out


def value_of_any(e):
    found = []
    e.as_scalar(lambda s: found.append(s.get()) or s)
    return found[0] if found else None


def walk(c):
    """Each position of a dense or sparse container, with its element's scalar, by its iterator."""
    out = []

    def visit(it):
        it.first()
        while it.index() is not None:
            values = []
            it.as_item(lambda e: e.as_scalar(lambda s: values.append(s.get()) or s))
            out.append((it.index(), values[0]))
            it.next()
    c.as_ordered(visit)
    return out

## CNT-01 · Elements by index: written through a builder's container, read through a data instance's; reads never raise

In [ ]:
card = (store.Card()
        .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a") and t.at(2).as_scalar("c"))))
        .phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.at("home").as_object(lambda r: r.number("1")))))
        .grid(lambda p: p.as_value(lambda a: a.as_container(lambda g: g.at((1, 2)).as_scalar(3.5))))
        .create())
tags, phones, grid = container(card.tags), container(card.phones), container(card.grid)
assert tags.as_schema() is Tags and (len(tags), len(phones), len(grid)) == (3, 1, 6)  # dense: every index up to the last
assert scalars(tags) == [(0, "a"), (2, "c")] and value_of_any(tags.at(1)) is None  # an index with nothing reads as empty
assert walk(tags) == [(0, "a"), (1, None), (2, "c")]  # an iterator visits every position of a dense container
assert walk(grid) == [((0, 0), None), ((0, 1), None), ((0, 2), None), ((1, 0), None), ((1, 1), None), ((1, 2), 3.5)]
seen = []
phones.as_dict(lambda d: seen.append(sorted(d)))
tags.as_dict(lambda d: seen.append("not a dict"))  # not called: tags is dense
phones.as_ordered(lambda it: seen.append("not ordered"))  # not called: phones is a dict
assert seen == [["home"]]
assert str(phones.at("home").as_object(lambda r: seen.append(str(r.number)) or r) and seen[-1]) == "1"
assert container(store.Card().create().tags) is None  # not set
card.name.as_value(lambda a: a.as_container(lambda c: seen.append((c.as_schema(), len(c), c.at(0).as_schema()))))
assert seen[-1] == (None, 0, None)  # a value of another kind: an empty visitor, read as an empty container
card.name.as_value(lambda a: a.as_container(lambda c: c.at(0).as_container(lambda x: seen.append(x.as_schema()))))
assert seen[-1] is None
empty = []
card.name.as_value(lambda a: a.as_container(lambda c: c.as_entries(lambda pairs: empty.append(list(pairs))).as_ordered(
    lambda it: empty.append("x")).as_dict(lambda d: empty.append("x"))))
assert empty == [[]]
with raises(ReadOnlyError, match="the container is read-only; write through a builder"):
    tags.remove(0)
with raises(ReadOnlyError, match="the container[0] is read-only; write through a builder"):
    tags.at(0).as_scalar("z")
with raises(ReadOnlyError, match="property 'tags' is read-only; write through a builder"):
    card.tags.as_value(lambda a: a.as_container(tags))
with raises(ReadOnlyError, match="property 'name' does not hold a container"):
    store.Card().name(lambda p: p.as_value(lambda a: a.as_container(tags)))
assert repr(tags) == "<container of 2>"
# A builder edits a copy, which it reads back; update() writes it, clone() copies it.
edited = store.Card(card).tags(lambda p: p.as_value(lambda a: a.as_container(
    lambda t: t.remove(2).at(1).as_scalar("b") and len(t) == 2 and t))).update()
assert edited is card and scalars(container(card.tags)) == [(0, "a"), (1, "b")]
copy = store.Card(card).clone()
assert scalars(container(copy.tags)) == [(0, "a"), (1, "b")] and container(copy.tags) is not container(card.tags)
# A container given as a value is written as it is; one of another schema is refused.
given = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(container(card.tags)))).create()
assert scalars(container(given.tags)) == [(0, "a"), (1, "b")]
with raises(TypeError, match="property 'tags' takes a container of its schema or a Spec, got _ContainerData"):
    store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(container(card.grid))))
assert scalars(container(store.Card().tags(container(card.tags)).create().tags)) == [(0, "a"), (1, "b")]  # as a property's value
twice = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a")))).tags(
    lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(1).as_scalar("b")))).create()
assert scalars(container(twice.tags)) == [(0, "a"), (1, "b")]  # a builder reads back, and adds to, what it has written
# Containers nest: a container's element may be a container.
nested = store.Card().rows(lambda p: p.as_value(lambda a: a.as_container(
    lambda r: r.at(1).as_container(lambda t: t.at(0).as_scalar("x"))))).create()
inner = []
container(nested.rows).at(1).as_container(lambda t: inner.append(scalars(t)))
assert inner == [[(0, "x")]] and len(container(nested.rows)) == 2

## CNT-02 · Iterators: positions before the first, at each, and past the last; a builder's extends a container and inserts into one of int indices

In [ ]:
sparse = store.Card().sparse(lambda p: p.as_value(lambda a: a.as_container(
    lambda c: c.at(10).as_scalar("j") and c.at(11).as_scalar("k") and c.at(40).as_scalar("z")))).create()
assert walk(container(sparse.sparse)) == [(10, "j"), (11, "k"), (40, "z")] and len(container(sparse.sparse)) == 3
moves = []
container(sparse.sparse).as_ordered(lambda it: moves.extend([
    it.index(), it.first().index(), it.next().index(), it.last().index(), it.next().index(), it.next().index(),
    it.prev().index(), it.first().prev().index(), it.prev().index(), repr(it)]))
assert moves == [None, 10, 11, 40, None, None, 40, None, None, "<iterator at None>"]
calls = []
container(sparse.sparse).as_ordered(lambda it: it.last().next().as_item(lambda e: calls.append(e)))
assert calls == []  # past the end of a read-only container: not called
with raises(ReadOnlyError, match="the container is read-only; write through a builder"):
    container(sparse.sparse).as_ordered(lambda it: it.first().prepend("x"))


def edit(card, name, cb):
    """Edits a card's container through its builder's iterator."""
    return getattr(store.Card(card), name)(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(cb)))).update()


edit(sparse, "sparse", lambda it: it.last().next().as_item(lambda e: e.as_scalar("past")).first().prev().as_item(
    lambda e: e.as_scalar("before")))
assert walk(container(sparse.sparse)) == [(10, "before"), (11, "j"), (12, "k"), (41, "z"), (42, "past")]
edit(sparse, "sparse", lambda it: it.first().append("after").prepend(lambda e: e.as_scalar("in")))
assert walk(container(sparse.sparse)) == [(10, "before"), (11, "in"), (12, "after"), (13, "j"), (14, "k"), (43, "z"), (44, "past")]
fresh = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(
    lambda it: it.as_item(lambda e: e.as_scalar("first")).next().as_item(lambda e: e.as_scalar("second")))))).create()
assert walk(container(fresh.tags)) == [(0, "first"), (1, "second")]  # an empty container extended from its ends
edit(fresh, "tags", lambda it: it.last().next().append("end").first().prev().prepend("start"))
assert walk(container(fresh.tags)) == [(0, "start"), (1, "first"), (2, "second"), (3, "end")]
blank = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(
    lambda it: it.append("only"))))).create()
assert walk(container(blank.tags)) == [(0, "only")]
ends = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(
    lambda it: it.first().as_item(lambda e: e.as_scalar("a")).last().next().prepend("z").first().prev().append("y"))))).create()
assert walk(container(ends.tags)) == [(0, "y"), (1, "a"), (2, "z")]  # at the ends, of an empty container and of one with elements
lone = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(lambda it: it.prepend("p"))))).create()
assert walk(container(lone.tags)) == [(0, "p")]
gridded = store.Card(card).grid(lambda p: p.as_value(lambda a: a.as_container(lambda g: g.as_ordered(
    lambda it: it.first().as_item(lambda e: e.as_scalar(0.5)))))).update()
assert walk(container(gridded.grid))[0] == ((0, 0), 0.5)
with raises(ValueError, match="property 'grid': only a dense or sparse container of int indices has positions to insert at; write through at(index)"):
    edit(card, "grid", lambda it: it.first().append(1.0))

## CNT-03 · Value objects in containers belong to the object holding the container: edited in place, copied by a clone, removed with their element

In [ ]:
owner = store.Card().phones(lambda p: p.as_value(lambda a: a.as_container(
    lambda d: d.at("home").as_object(lambda r: r.number("1").label("h")) and d.at("work").as_object(lambda r: r.number("2"))))).create()
held = {}
container(owner.phones).as_dict(lambda d: held.update({k: e for k, e in d.items()}))
home = []
held["home"].as_object(lambda r: home.append(r) or r)
assert Proxies.owner_of(home[0]) is owner and str(home[0].label) == "h"
store.Card(owner).phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.at("home").as_object(lambda r: r.label("x"))))).update()
again = []
container(owner.phones).at("home").as_object(lambda r: again.append(r) or r)
assert again[0] is home[0] and str(home[0].label) == "x" and str(home[0].number) == "1"  # edited in place
twin = store.Card(owner).clone()
copied = []
container(twin.phones).at("home").as_object(lambda r: copied.append(r) or r)
assert copied[0] is not home[0] and Proxies.owner_of(copied[0]) is twin and str(copied[0].number) == "1"
store.Card(owner).phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.remove("home")))).update()
assert len(container(owner.phones)) == 1 and Proxies.owner_of(home[0]) is owner
Kit = S.OfObject.Builder().properties(lambda p: p.name("parts").of(Tags)).create()  # a value object holding a container
Box = S.OfObject.Builder().name("Box").ref().properties(lambda p: p.name("kit").of(Kit)).create()
store.register(Box)
box = store.Box().kit(lambda p: p.as_object(lambda k: k.parts(lambda q: q.as_value(lambda a: a.as_container(
    lambda t: t.at(0).as_scalar("bolt")))))).create()
twin_box = store.Box(box).clone()
parts = []
twin_box.kit.as_object(lambda k: (k.parts.as_value(lambda a: a.as_container(lambda t: parts.append(scalars(t)))), k)[1])
assert parts == [[(0, "bolt")]]  # copied with the value object that holds it
moved = store.Card().tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a")))).create()
store.Card(moved).tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.as_ordered(lambda it: it.first().prepend("z"))))).update()
assert walk(container(moved.tags)) == [(0, "z"), (1, "a")]

## CNT-04 · Snapshots: a dense container as an array, nested or flat for several indices, a sparse one as blocks, a dict by text as a mapping, entries otherwise; read back, and over what a store holds

In [ ]:
card = (store.Card()
        .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a") and t.at(2).as_scalar("c"))))
        .phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.at("home").as_object(lambda r: r.number("1")))))
        .grid(lambda p: p.as_value(lambda a: a.as_container(lambda g: g.at((1, 2)).as_scalar(3.5) and g.at((0, 0)).as_scalar(0.25))))
        .sparse(lambda p: p.as_value(lambda a: a.as_container(lambda s: s.at(3).as_scalar("x") and s.at(4).as_scalar("y") and s.at(9).as_scalar("z"))))
        .rows(lambda p: p.as_value(lambda a: a.as_container(lambda r: r.at(0).as_container(lambda t: t.at(1).as_scalar("n")))))
        .create())
written = root_of(Plain.ToPlain(store).OfObject(Card, card))
assert len(container(card.grid)) == 6  # the box its indices span
assert written == {"tags": ["a", None, "c"], "phones": {"home": {"number": "1"}}, "grid": [[0.25, None, None], [None, None, 3.5]],
                   "sparse": [{"at": 3, "items": ["x", "y"]}, {"at": 9, "items": ["z"]}], "rows": [[None, "n"]]}
flat = root_of(Plain.ToPlain(store, flat=True).OfObject(Card, card))
assert flat["grid"] == {"shape": [2, 3], "items": [0.25, None, None, None, None, 3.5]}
assert '"grid": {"shape": [2, 3]' in JSON.ToJSON(store, flat=True).OfObject(Card, card) and "shape:" in YAML.ToYAML(store, flat=True).OfObject(Card, card)
other = Proxies.OfStore()
other.register(Card)
for snapshot in (Plain.ToPlain(store).OfObject(Card, card), Plain.ToPlain(store, flat=True).OfObject(Card, card)):
    back = Plain.FromPlain(other).OfObject(Card, snapshot)  # the loader takes either form
    assert root_of(Plain.ToPlain(other).OfObject(Card, back)) == written
assert Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) is card  # reloading: an update
assert root_of(Plain.ToPlain(store).OfObject(Card, card)) == written
# Any other index is written as entry-like elements: a dict by bytes, a dense container by text, a sparse one by text.
ByBytes = S.OfContainer.Builder().index(S.OfScalar.Data(bytes)).element(Int).create()
Sorted = S.OfContainer.Builder().index(Text).element(Int).dense().create()
Keyed = S.OfContainer.Builder().index(Cell).element(Int).sparse().create()
Mixed = S.OfObject.Builder().properties(text("k"), text("n", int)).create()
ByMixed = S.OfContainer.Builder().index(Mixed).element(Int).create()
Note = S.OfObject.Builder().name("Note").ref().properties(lambda p: p.name("b").of(ByBytes), lambda p: p.name("s").of(Sorted),
                                                          lambda p: p.name("k").of(Keyed), lambda p: p.name("m").of(ByMixed)).create()
store.register(Note)
note = (store.Note().b(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(b"\x01").as_scalar(1))))
        .s(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at("b").as_scalar(2) and c.at("a").as_scalar(1))))
        .k(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at((0, 1)).as_scalar(1) and c.at((0, 2)).as_scalar(2) and c.at((3, 0)).as_scalar(3))))
        .m(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(("x", 1)).as_scalar(5)))).create())
noted = root_of(Plain.ToPlain(store).OfObject(Note, note))
assert noted == {"b": [{"index": "AQ==", "element": 1}], "s": [{"index": "a", "element": 1}, {"index": "b", "element": 2}],
                 "k": [{"at": {"row": 0, "col": 1}, "items": [1, 2]}, {"at": {"row": 3, "col": 0}, "items": [3]}],
                 "m": [{"index": {"k": "x", "n": 1}, "element": 5}]}
other.register(Note)
assert root_of(Plain.ToPlain(other).OfObject(Note, Plain.FromPlain(other).OfObject(Note, Plain.ToPlain(store).OfObject(Note, note)))) == noted
# A union's branch, or an intersection's part, may hold a container.
Either = S.OfUnion.Builder().branches(lambda b: b.name("tags").of(Tags), lambda b: b.name("word").of(Text)).create()
Choice = S.OfObject.Builder().name("Choice").ref().properties(lambda p: p.name("pick").of(Either)).create()
store.register(Choice)
other.register(Choice)
choice = store.Choice().pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(lambda t: t.at(1).as_scalar("q"))))).create()
picked = root_of(Plain.ToPlain(store).OfObject(Choice, choice))
assert picked == {"pick": {"tags": [None, "q"]}}
assert root_of(Plain.ToPlain(other).OfObject(Choice, Plain.FromPlain(other).OfObject(Choice, Plain.ToPlain(store).OfObject(Choice, choice)))) == picked
store.Choice(choice).pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(lambda t: t.at(2).as_scalar("r"))))).update()
assert root_of(Plain.ToPlain(store).OfObject(Choice, choice)) == {"pick": {"tags": [None, "q", "r"]}}  # its container edited
same_branch = store.Choice().pick(lambda p: p.as_value(lambda a: a.as_union(
    lambda u: u.tags(lambda t: t.at(0).as_scalar("p")).tags(lambda t: t.at(1).as_scalar("q"))))).create()  # read back, then written again
assert root_of(Plain.ToPlain(store).OfObject(Choice, same_branch)) == {"pick": {"tags": ["p", "q"]}}
given = store.Choice().pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(container(card.tags))))).create()
assert root_of(Plain.ToPlain(store).OfObject(Choice, given)) == {"pick": {"tags": ["a", None, "c"]}}  # a container given
with raises(TypeError, match="branch 'tags' takes a container of its schema or a Spec, got int"):
    store.Choice().pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(1))))
with raises(ReadOnlyError, match="the container is read-only; write through a builder"):
    container(card.tags)(lambda t: t)
# An entry's property may hold a container: entries equal by their containers are one, in a set.
Marks = S.OfObject.Builder().name("Marks").ref().relations(lambda r: r.name("sets").me("owner").of(
    lambda rel: rel.links("owner").properties(lambda p: p.name("tags").of(Tags), lambda p: p.name("grid").of(Grid)).unique())).create()
store.register(Marks)
other.register(Marks)
two = lambda t: t.at(0).as_scalar("a") and t.at(1).as_scalar("b")
cell = lambda g: g.at((0, 1)).as_scalar(1.5)
marks = store.Marks().sets(lambda adj: adj.insert(lambda e: e.tags(lambda p: p.as_value(lambda a: a.as_container(two))).grid(
    lambda p: p.as_value(lambda a: a.as_container(cell)))).insert(lambda e: e.tags(lambda p: p.as_value(lambda a: a.as_container(two))).grid(
    lambda p: p.as_value(lambda a: a.as_container(cell))))).create()
assert len(marks.sets) == 1 and root_of(Plain.ToPlain(store).OfObject(Marks, marks)) == {"sets": [{"tags": ["a", "b"], "grid": [[None, 1.5]]}]}
doubled = {"root": "Marks/m", "objects": {"Marks/m": {"sets": [{"tags": ["a", "b"], "grid": [[None, 1.5]]},
                                                              {"tags": ["a", "b"], "grid": [[None, 1.5]]}]}}}
assert len(Plain.FromPlain(other).OfObject(Marks, doubled).sets) == 1  # read once, by value
assert len(container(note.s)) == 2  # a dense container by text: the elements present
emptied = Plain.FromPlain(other).OfObject(Note, {"root": "Note/e", "objects": {"Note/e": {"b": [{"index": "AQ=="}], "s": []}}})
assert root_of(Plain.ToPlain(other).OfObject(Note, emptied)) == {"b": [], "s": []}  # an element absent; empty containers
branch = []
choice.pick.as_value(lambda a: a.as_union(lambda u: branch.append(u.tags) or u))
store.Choice().create().pick.as_value(lambda a: a.as_union(lambda u: branch.append(u.tags) or u))
assert len(branch[0]) == 3 and branch[1].as_schema() is None  # a union value not set: its branch reads as empty

## CNT-05 · Malformed containers in snapshots are refused, with a path

In [ ]:
def refused(name, value, message):
    with raises(DecodeError, match=message):
        Plain.FromPlain(store).OfObject(Card, {"root": "Card/card-900", "objects": {"Card/card-900": {name: value}}})


at = '$.objects["Card/card-900"]'
refused("tags", {"a": 1}, f"{at}.tags: a dense container must be a list, got dict")
refused("tags", [1], f"{at}.tags[0]: expected str, got int")
refused("grid", [[1.0], 2.0], f"{at}.grid[1]: a container of 2 indices nests 2 lists")
refused("grid", {"shape": [2], "items": [1.0]}, f"{at}.grid: a flat container's shape is 2 sizes, and its items as many as they make")
refused("grid", {"shape": [1, 2], "items": [1.0]}, f"{at}.grid: a flat container's shape is 2 sizes")
refused("grid", {"shape": "2, 3", "items": []}, f"{at}.grid: a flat container's shape is 2 sizes")
refused("grid", {"shape": [1, 1], "items": [True]}, f"{at}.grid.items[0]: expected float, got bool")
refused("sparse", {"at": 1}, f"{at}.sparse: a sparse container must be a list of blocks, got dict")
refused("sparse", [{"at": 1}], f"{at}.sparse[0]: a block is {{'at': index, 'items': [element, ...]}}")
refused("sparse", [{"at": "1", "items": []}], f"{at}.sparse[0].at: expected int, got str")
refused("phones", [], f"{at}.phones: a container keyed by text must be a mapping, got list")
refused("phones", {"home": 1}, f"{at}.phones.home: a value object must be a mapping, got int")
with raises(DecodeError, match='$.objects["Note/n"].m: a container must be a list, got dict'):
    Plain.FromPlain(store).OfObject(Note, {"root": "Note/n", "objects": {"Note/n": {"m": {}}}})
with raises(DecodeError, match="""$.objects["Note/n"].m[0]: an entry of a container is {'index': index, 'element': element}"""):
    Plain.FromPlain(store).OfObject(Note, {"root": "Note/n", "objects": {"Note/n": {"m": [{"element": 1}]}}})
with raises(DecodeError, match="""$.objects["Note/n"].k[0].at: an index has every property of its value object; ['col'] are not set"""):
    Plain.FromPlain(store).OfObject(Note, {"root": "Note/n", "objects": {"Note/n": {"k": [{"at": {"row": 1}, "items": [1]}]}}})
Deep = S.OfObject.Builder().properties(lambda p: p.name("cell").of(Cell)).create()
Odd = S.OfContainer.Builder().index(Deep).element(Int).create()  # entries: its index is neither ordered nor hashable
Rare = S.OfObject.Builder().name("Rare").ref().properties(lambda p: p.name("odd").of(Odd)).create()
store.register(Rare)
with raises(DecodeError, match="""$.objects["Rare/r"].odd[0].index: a container whose index is neither ordered nor hashable is not read from snapshots yet"""):
    Plain.FromPlain(store).OfObject(Rare, {"root": "Rare/r", "objects": {"Rare/r": {"odd": [{"index": {}, "element": 1}]}}})

## CNT-06 · An index is checked against the index type: a scalar of its type, or a tuple of a value object index's properties; a dense container's is not negative

In [ ]:
def at(name, index):
    getattr(store.Card(), name)(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(index))))


with raises(TypeError, match="property 'tags': an index is a int, got str"):
    at("tags", "0")
with raises(TypeError, match="property 'tags': an index is a int, got bool"):
    at("tags", True)
with raises(TypeError, match="property 'grid': an index is a tuple of int, int, got [1]"):
    at("grid", (1,))
with raises(TypeError, match="property 'grid': an index is a tuple of int, int, got [1, '2']"):
    at("grid", (1, "2"))
with raises(ValueError, match="property 'tags': a dense container's index is not negative, got -1"):
    at("tags", -1)
with raises(ValueError, match="property 'grid': a dense container's index is not negative, got [0, -1]"):
    at("grid", (0, -1))
store.Card().sparse(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(-5).as_scalar("ok"))))  # sparse: any int
deep = []
store.Rare().odd(lambda p: p.as_value(lambda a: a.as_container(lambda c: deep.append(c))))
with raises(TypeError, match="property 'odd': an index is a value of its index type, got int"):
    deep[0].at(1)
# An index of a value object that is neither ordered nor hashable: given as a value object, written as one.
Keyed = S.OfObject.Builder().name("Keyed").ref().properties(lambda p: p.name("key").of(Deep), lambda p: p.name("odd").of(Odd)).create()
store.register(Keyed)
keyed = store.Keyed().key(lambda p: p.as_object(lambda k: k.cell(lambda q: q.as_object(lambda c: c.row(1).col(2))))).create()
key = []
keyed.key.as_object(lambda k: key.append(k) or k)
keyed = store.Keyed(keyed).odd(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(key[0]).as_scalar(7)))).update()
assert root_of(Plain.ToPlain(store).OfObject(Keyed, keyed))["odd"] == [{"index": {"cell": {"row": 1, "col": 2}}, "element": 7}]